# 3D Machine Learning Visualization Suite

A reproducible end-to-end classification workflow featuring 3D/interactive visualizations, model comparison, hyperparameter tuning, and explainability.

**Dataset:** scikit-learn Breast Cancer Wisconsin (Diagnostic), loaded locally from scikit-learn (no download required). This is a demonstration/educational workflow, not a clinical diagnostic tool.

**Optional packages:** `xgboost` and `shap`. The notebook detects their availability and continues without them.

Now also includes Pandas data engineering, an optional TensorFlow dense network, and a CNN-based computer-vision example using MNIST.

## 1. Environment and imports

In [ ]:
# If needed, uncomment and run this once in your environment:
# %pip install -U numpy pandas matplotlib seaborn scikit-learn plotly nbformat
# %pip install xgboost shap tensorflow

import warnings, time, sys, importlib.util
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "figure.dpi": 110})
HAS_PLOTLY = importlib.util.find_spec("plotly") is not None
HAS_XGBOOST = importlib.util.find_spec("xgboost") is not None
HAS_SHAP = importlib.util.find_spec("shap") is not None
print("Python:", sys.version.split()[0], "| Plotly:", HAS_PLOTLY, "| XGBoost:", HAS_XGBOOST, "| SHAP:", HAS_SHAP)

## 2. Load and inspect data

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()  # 0 = malignant, 1 = benign
class_names = list(dataset.target_names)
print("Features:", X.shape[1], "Samples:", X.shape[0], "Classes:", class_names)
display(X.head())
display(y.value_counts().rename(index={0:"malignant", 1:"benign"}).to_frame("count"))

In [ ]:
# Basic data quality checks
quality = pd.DataFrame({"dtype": X.dtypes.astype(str), "missing": X.isna().sum(), "n_unique": X.nunique()})
display(quality)
print("Duplicate rows:", X.duplicated().sum())

## 3. Train/test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
# Scale for linear/MLP models; tree models are also placed in pipelines for a consistent interface.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 4. Train baseline models

In [ ]:
models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(n_estimators=350, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=350, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=180, learning_rate=0.08, random_state=RANDOM_STATE),
    "Neural Network": Pipeline([("scale", StandardScaler()), ("model", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=700, early_stopping=True, random_state=RANDOM_STATE))])
}
if HAS_XGBOOST:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)

fitted_models, rows = {}, []
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    fitted_models[name] = model
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred), "ROC AUC": roc_auc_score(y_test, proba) if proba is not None else np.nan,
                 "Fit time (s)": elapsed})
results = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
display(results.style.format({c:"{:.3f}" for c in results.select_dtypes("number").columns}))

## 5. Model comparison visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC AUC"], var_name="Metric", value_name="Score"),
            x="Model", y="Score", hue="Metric", ax=ax)
ax.set_ylim(0, 1.05); ax.tick_params(axis="x", rotation=25); ax.set_title("Held-out test performance")
plt.tight_layout(); plt.show()

In [ ]:
best_name = results.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
print("Best model by test ROC AUC:", best_name)
print(classification_report(y_test, best_pred, target_names=class_names))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=class_names, cmap="Blues")
plt.title(f"Confusion matrix — {best_name}"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in fitted_models.items():
    if hasattr(model, "predict_proba"):
        RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0,1], [0,1], "k--", alpha=.5); ax.set_title("ROC curves"); plt.tight_layout(); plt.show()

## 6. PCA projections and 3D decision-space visualization

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca2 = pca2.fit_transform(X_train_scaled)
X_test_pca2 = pca2.transform(X_test_scaled)
print("2D explained variance:", pca2.explained_variance_ratio_.sum())
plt.figure(figsize=(8,6))
plt.scatter(X_test_pca2[:,0], X_test_pca2[:,1], c=y_test, cmap="coolwarm", edgecolor="k", alpha=.8)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Test data projected onto first two principal components")
plt.colorbar(label="Class (0 malignant, 1 benign)"); plt.show()

In [ ]:
pca3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_all_3d = pca3.fit_transform(StandardScaler().fit_transform(X))
fig = plt.figure(figsize=(9,7)); ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(X_all_3d[:,0], X_all_3d[:,1], X_all_3d[:,2], c=y, cmap="coolwarm", s=22, alpha=.75)
ax.set(xlabel="PC1", ylabel="PC2", zlabel="PC3", title="3D PCA projection")
fig.colorbar(sc, ax=ax, label="Class"); plt.show()
print("3D explained variance:", pca3.explained_variance_ratio_.sum())

In [ ]:
# 2D PCA decision boundaries for representative classifiers (visualization only).
from sklearn.inspection import DecisionBoundaryDisplay
boundary_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(n_estimators=150, random_state=RANDOM_STATE),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, early_stopping=True, random_state=RANDOM_STATE)
}
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (name, clf) in zip(axes.ravel(), boundary_models.items()):
    clf.fit(X_train_pca2, y_train)
    DecisionBoundaryDisplay.from_estimator(clf, X_train_pca2, response_method="predict", alpha=.25,
        cmap="coolwarm", ax=ax, grid_resolution=120)
    ax.scatter(X_train_pca2[:,0], X_train_pca2[:,1], c=y_train, cmap="coolwarm", edgecolor="k", s=12)
    ax.set_title(name); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

## 7. Interactive 3D plots with Plotly

In [ ]:
if HAS_PLOTLY:
    import plotly.express as px
    plot_df = pd.DataFrame(X_all_3d, columns=["PC1", "PC2", "PC3"])
    plot_df["Class"] = pd.Series(y).map({0:"malignant", 1:"benign"}).values
    fig = px.scatter_3d(plot_df, x="PC1", y="PC2", z="PC3", color="Class",
                        title="Interactive 3D PCA — rotate, zoom, and hover",
                        opacity=.75, hover_data=["Class"])
    fig.show()
else:
    print("Plotly is not installed. Run: %pip install plotly")

In [ ]:
if HAS_PLOTLY:
    metric_long = results.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC AUC"], var_name="Metric", value_name="Score")
    px.bar(metric_long, x="Model", y="Score", color="Metric", barmode="group",
           title="Interactive model metric comparison", range_y=[0,1.05]).show()

## 8. Feature importance and model explainability

In [ ]:
# Native feature importance where supported.
importance_model = fitted_models.get("Random Forest", best_model)
if hasattr(importance_model, "feature_importances_"):
    imp = pd.Series(importance_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
else:
    perm = permutation_importance(importance_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1)
    imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(20)
imp.sort_values().plot(kind="barh", figsize=(9,7), title="Top 20 feature importances")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()

In [ ]:
# Model-agnostic permutation importance for the selected best model.
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring="roc_auc")
perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(15)
display(perm_imp.rename("Mean ROC-AUC decrease").to_frame())
perm_imp.sort_values().plot(kind="barh", figsize=(8,6), title=f"Permutation importance — {best_name}")
plt.tight_layout(); plt.show()

In [ ]:
# SHAP is optional. TreeExplainer is used for tree ensembles; otherwise KernelExplainer is a slower fallback.
if HAS_SHAP:
    import shap
    tree_candidate = fitted_models.get("XGBoost", fitted_models.get("Random Forest", fitted_models.get("Extra Trees")))
    try:
        explainer = shap.TreeExplainer(tree_candidate)
        shap_values = explainer.shap_values(X_test.iloc[:100])
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]
        elif getattr(shap_values, "ndim", 0) == 3:
            shap_values = shap_values[:, :, -1]
        shap.summary_plot(shap_values, X_test.iloc[:100], show=True, max_display=15)
    except Exception as exc:
        print("SHAP tree explanation unavailable:", repr(exc))
        print("Permutation importance above remains available.")
else:
    print("SHAP is optional and not installed. Run %pip install shap, then rerun this cell.")
    print("Permutation importance above provides a model-agnostic explanation.")

In [ ]:
# SHAP interaction values for XGBoost, when both package and model are available.
if HAS_SHAP and "XGBoost" in fitted_models:
    try:
        xgb_model = fitted_models["XGBoost"]
        shap_interaction_values = shap.TreeExplainer(xgb_model).shap_interaction_values(X_test.iloc[:60])
        print("SHAP interaction tensor shape:", np.asarray(shap_interaction_values).shape)
        # Mean absolute interaction strength (excluding diagonal self-effects).
        inter = np.abs(np.asarray(shap_interaction_values))
        if inter.ndim == 4: inter = inter[:, :, :, -1]
        mean_inter = inter.mean(axis=0)
        np.fill_diagonal(mean_inter, 0)
        top_idx = np.argsort(mean_inter.sum(axis=0))[-12:]
        sns.heatmap(mean_inter[np.ix_(top_idx, top_idx)], xticklabels=X.columns[top_idx], yticklabels=X.columns[top_idx], cmap="viridis")
        plt.title("Mean absolute SHAP feature interactions (top features)"); plt.xticks(rotation=75); plt.tight_layout(); plt.show()
    except Exception as exc:
        print("SHAP interaction analysis could not run:", repr(exc))
else:
    print("SHAP interaction analysis requires SHAP and an available XGBoost model.")

## 9. Hyperparameter tuning

In [ ]:
# Randomized search uses stratified CV and a compact search space to keep runtime manageable.
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={"n_estimators":[150,250,400], "max_depth":[None,4,7,10],
                         "min_samples_split":[2,5,10], "min_samples_leaf":[1,2,4],
                         "max_features":["sqrt", 0.5]},
    n_iter=12, scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)
tuned_rf = rf_search.best_estimator_
print("Best CV ROC AUC:", round(rf_search.best_score_, 4))
print("Best parameters:", rf_search.best_params_)
print("Tuned test ROC AUC:", round(roc_auc_score(y_test, tuned_rf.predict_proba(X_test)[:,1]), 4))

In [ ]:
# Optional MLP hyperparameter comparison (small, reproducible search).
mlp_pipe = Pipeline([("scale", StandardScaler()), ("model", MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE))])
mlp_search = RandomizedSearchCV(mlp_pipe, {
    "model__hidden_layer_sizes":[(32,), (64,), (64,32)],
    "model__alpha":[0.0001, 0.001, 0.01], "model__learning_rate_init":[0.0005, 0.001, 0.005]
}, n_iter=6, scoring="roc_auc", cv=3, random_state=RANDOM_STATE, n_jobs=-1)
mlp_search.fit(X_train, y_train)
print("Best MLP CV ROC AUC:", round(mlp_search.best_score_, 4))
print("Best MLP parameters:", mlp_search.best_params_)

In [ ]:
# Cross-validation scores for the tuned forest (training set only; test set remains held out).
cv_scores = cross_val_score(tuned_rf, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="roc_auc", n_jobs=-1)
sns.boxplot(x=cv_scores); plt.title("Tuned Random Forest: 5-fold ROC-AUC"); plt.xlabel("ROC AUC"); plt.show()
print("Fold scores:", np.round(cv_scores, 4), "Mean:", round(cv_scores.mean(), 4))

## 10. 3D hyperparameter and training diagnostics

In [ ]:
# Actual grid of Random Forest hyperparameters: each point is evaluated on a validation split.
from sklearn.model_selection import ParameterGrid
X_sub_train, X_val, y_sub_train, y_val = train_test_split(X_train, y_train, test_size=.22, stratify=y_train, random_state=RANDOM_STATE)
depths = [2, 4, 6, 10, None]
n_estimators_grid = [50, 100, 180]
records = []
for depth in depths:
    for n_est in n_estimators_grid:
        candidate = RandomForestClassifier(max_depth=depth, n_estimators=n_est, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
        candidate.fit(X_sub_train, y_sub_train)
        score = roc_auc_score(y_val, candidate.predict_proba(X_val)[:,1])
        records.append({"max_depth": "None" if depth is None else str(depth), "depth_numeric": 14 if depth is None else depth,
                        "n_estimators": n_est, "validation_auc": score})
hyper_df = pd.DataFrame(records)
display(hyper_df.pivot(index="max_depth", columns="n_estimators", values="validation_auc"))
if HAS_PLOTLY:
    px.scatter_3d(hyper_df, x="depth_numeric", y="n_estimators", z="validation_auc", color="validation_auc",
                  hover_data=["max_depth"], title="3D Random Forest hyperparameter validation surface").show()
else:
    fig = plt.figure(figsize=(8,6)); ax = fig.add_subplot(111, projection="3d")
    ax.scatter(hyper_df.depth_numeric, hyper_df.n_estimators, hyper_df.validation_auc, c=hyper_df.validation_auc, cmap="viridis")
    ax.set(xlabel="max_depth (None shown as 14)", ylabel="n_estimators", zlabel="Validation ROC AUC"); plt.show()

In [ ]:
# MLP training loss curve from the fitted baseline network.
mlp = fitted_models["Neural Network"].named_steps["model"]
plt.plot(mlp.loss_curve_); plt.xlabel("Iteration"); plt.ylabel("Loss"); plt.title("Neural Network training loss"); plt.show()

## 11. Quantization / model-size trade-off (illustrative calculation)

In [ ]:
# Illustrative only: assumes 50 MB of 32-bit weights and a hypothetical accuracy penalty.
bits = np.array([1, 2, 4, 8, 16, 32])
size_mb = 50 * bits / 32
acc_drop = np.array([9.2, 4.5, 1.2, 0.3, 0.05, 0.0])
quant_df = pd.DataFrame({"Bits per weight":bits, "Estimated size (MB)":size_mb, "Illustrative accuracy drop (pp)":acc_drop})
display(quant_df)
fig, ax1 = plt.subplots(figsize=(8,5)); ax1.plot(bits, size_mb, "o-", label="Size (MB)"); ax1.set_xscale("log", base=2)
ax1.set_xlabel("Bits per weight"); ax1.set_ylabel("Estimated model size (MB)")
ax2 = ax1.twinx(); ax2.plot(bits, acc_drop, "s--", label="Accuracy drop"); ax2.set_ylabel("Illustrative accuracy drop (percentage points)")
plt.title("Quantization size–accuracy trade-off (hypothetical)"); fig.tight_layout(); plt.show()

## 12. Prediction helper and final evaluation

In [ ]:
def predict_one(sample: pd.DataFrame, model=best_model):
    """Predict a single or batch of rows with the fitted feature schema."""
    if not isinstance(sample, pd.DataFrame):
        raise TypeError("sample must be a pandas DataFrame with the original feature columns")
    missing = set(X.columns) - set(sample.columns)
    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")
    sample = sample.loc[:, X.columns]
    labels = model.predict(sample)
    probabilities = model.predict_proba(sample) if hasattr(model, "predict_proba") else None
    output = pd.DataFrame({"predicted_class": pd.Series(labels, index=sample.index).map({0:"malignant",1:"benign"})}, index=sample.index)
    if probabilities is not None:
        output["P(malignant)"] = probabilities[:,0]
        output["P(benign)"] = probabilities[:,1]
    return output

display(predict_one(X_test.iloc[:5]))

In [ ]:
# Compare baseline winner with tuned Random Forest on the untouched test set.
final_candidates = {best_name: best_model, "Tuned Random Forest": tuned_rf}
final_rows = []
for name, model in final_candidates.items():
    pred = model.predict(X_test); prob = model.predict_proba(X_test)[:,1]
    final_rows.append({"Model":name, "Accuracy":accuracy_score(y_test,pred), "F1":f1_score(y_test,pred), "ROC AUC":roc_auc_score(y_test,prob)})
final_results = pd.DataFrame(final_rows).drop_duplicates("Model")
display(final_results)
print("Workflow complete. All reported test metrics use the held-out test split.")

## Notes and reproducibility

- The notebook uses a fixed random seed (`RANDOM_STATE = 42`) and a stratified hold-out test set.
- Hyperparameter selection uses training data only; the test split is reserved for final evaluation.
- PCA plots are projections and can distort class separation; they are not substitutes for full-feature model evaluation.
- SHAP and XGBoost sections are optional and gracefully skip when dependencies are unavailable.
- The quantization section is explicitly hypothetical; it does not quantize a trained model or measure real deployment accuracy.
- For real deployment, validate on representative external data, assess calibration and subgroup performance, and obtain domain-expert review.

## 11. Pandas data engineering and exploratory analysis

This section demonstrates practical Pandas workflows on the current tabular dataset: profiling, missing-value handling, duplicate checks, grouped summaries, correlation analysis, and reproducible feature-table export.

In [ ]:
# Keep a single analysis table with features and target labels
df = X.copy()
df["target"] = y.to_numpy()
df["target_name"] = pd.Series(y, index=y.index).map(dict(enumerate(class_names)))

# Data-quality and summary reports
pandas_profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(pandas_profile)
display(df.describe(include="all").T.head(15))

print("Duplicate rows:", int(df.duplicated().sum()))
print("Class balance:")
display(df["target_name"].value_counts().to_frame("samples"))

# Example cleaning pipeline (does not modify the train/test data already created)
df_clean = df.drop_duplicates().copy()
numeric_cols = df_clean.select_dtypes(include=np.number).columns.drop("target", errors="ignore")
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
print("Cleaned shape:", df_clean.shape)

In [ ]:
# Correlation heatmap and class-wise feature summary
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.2)
plt.title("Pandas: numeric feature correlation matrix")
plt.tight_layout()
plt.show()

class_summary = df.groupby("target_name")[list(X.columns[:6])].agg(["mean", "median", "std"])
display(class_summary)

## 12. Deep learning with TensorFlow/Keras (optional)

A dense neural network is trained on the same stratified train/test split. TensorFlow is optional; if it is not installed, this section reports how to enable it and leaves the classical ML workflow usable. The test set is used only for final evaluation.

In [ ]:
HAS_TENSORFLOW = importlib.util.find_spec("tensorflow") is not None
if HAS_TENSORFLOW:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    dl_scaler = StandardScaler()
    X_dl_train = dl_scaler.fit_transform(X_train).astype("float32")
    X_dl_test = dl_scaler.transform(X_test).astype("float32")

    deep_model = keras.Sequential([
        layers.Input(shape=(X_dl_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1, activation="sigmoid")
    ])
    deep_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                       loss="binary_crossentropy", metrics=["accuracy", keras.metrics.AUC(name="auc")])
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                               restore_best_weights=True)
    history = deep_model.fit(X_dl_train, np.asarray(y_train), validation_split=0.2,
                             epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)
    dl_prob = deep_model.predict(X_dl_test, verbose=0).ravel()
    dl_pred = (dl_prob >= 0.5).astype(int)
    print(classification_report(y_test, dl_pred, target_names=class_names))
    print("Deep-learning ROC AUC:", round(roc_auc_score(y_test, dl_prob), 4))

    pd.DataFrame(history.history)[["loss", "val_loss"]].plot(title="Keras training and validation loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.show()
else:
    print("TensorFlow is not installed. To enable this section, run: %pip install tensorflow")
    print("The existing scikit-learn Neural Network model remains available in the baseline comparison.")

## 13. Computer vision: image classification with a CNN (optional)

This section uses the built-in MNIST handwritten-digit dataset from Keras. It trains a compact convolutional neural network (CNN), plots training curves, and displays sample predictions. It runs only when TensorFlow is installed and downloads MNIST on first use.

In [ ]:
if HAS_TENSORFLOW:
    (x_img_train, y_img_train), (x_img_test, y_img_test) = keras.datasets.mnist.load_data()
    x_img_train = (x_img_train.astype("float32") / 255.0)[..., np.newaxis]
    x_img_test = (x_img_test.astype("float32") / 255.0)[..., np.newaxis]

    cnn = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(10, activation="softmax")
    ])
    cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn_history = cnn.fit(x_img_train, y_img_train, validation_split=0.1,
                          epochs=5, batch_size=128, verbose=1)
    cv_loss, cv_accuracy = cnn.evaluate(x_img_test, y_img_test, verbose=0)
    print(f"MNIST CNN test accuracy: {cv_accuracy:.4f} | test loss: {cv_loss:.4f}")
else:
    print("Computer-vision CNN skipped: install TensorFlow with %pip install tensorflow")

In [ ]:
if HAS_TENSORFLOW:
    pd.DataFrame(cnn_history.history)[["accuracy", "val_accuracy"]].plot(title="CNN accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.show()

    sample_images = x_img_test[:12]
    sample_probs = cnn.predict(sample_images, verbose=0)
    sample_preds = sample_probs.argmax(axis=1)
    fig, axes = plt.subplots(3, 4, figsize=(9, 7))
    for i, ax in enumerate(axes.flat):
        ax.imshow(sample_images[i].squeeze(), cmap="gray")
        ax.set_title(f"Pred: {sample_preds[i]} | True: {y_img_test[i]}")
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 14. Pandas results registry and export

Collect the classical ML comparison and (when available) the deep-learning result in one Pandas table. Export is optional and writes a CSV into the notebook's current working directory.

In [ ]:
# Reuse the existing model comparison table
final_results = results.copy()
if "HAS_TENSORFLOW" in globals() and HAS_TENSORFLOW and "dl_prob" in globals():
    final_results = pd.concat([final_results, pd.DataFrame([{
        "Model": "TensorFlow Dense Network",
        "Accuracy": accuracy_score(y_test, dl_pred),
        "Precision": precision_score(y_test, dl_pred, zero_division=0),
        "Recall": recall_score(y_test, dl_pred, zero_division=0),
        "F1": f1_score(y_test, dl_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, dl_prob),
        "Fit time (s)": np.nan
    }])], ignore_index=True)

display(final_results.sort_values("ROC AUC", ascending=False))
# Uncomment to save a portable results file:
# final_results.to_csv("3d_ml_model_comparison.csv", index=False)

## Additional dependencies and execution notes

- Pandas is part of the core workflow and is used for profiling, cleaning, grouping, correlation, and exports.
- Deep learning and computer vision require TensorFlow: `%pip install tensorflow`.
- The CNN downloads MNIST the first time it runs; an internet connection may be required.
- Run notebook cells from top to bottom. Optional sections skip safely when TensorFlow is unavailable.
- The tabular breast-cancer dataset and MNIST image dataset are separate demonstrations; do not interpret the MNIST score as a medical-model result.
- This notebook is educational, not a clinical diagnostic system.

## 15. Advanced Pandas: feature engineering, reusable preprocessing, and data export

This section adds reproducible feature engineering and a reusable Pandas preparation function. It operates on a copy and does not alter the earlier train/test split. For external datasets, pass a CSV path to `load_tabular_csv`; the target column is kept separate to reduce accidental leakage.

In [ ]:
def load_tabular_csv(path, target_column, *, drop_duplicates=True):
    """Load a tabular CSV, separate target, and report basic data quality."""
    frame = pd.read_csv(path)
    if target_column not in frame.columns:
        raise KeyError(f"Target column {target_column!r} not found")
    if drop_duplicates:
        frame = frame.drop_duplicates().copy()
    y_external = frame.pop(target_column)
    X_external = frame.copy()
    report = pd.DataFrame({
        "dtype": X_external.dtypes.astype(str),
        "missing": X_external.isna().sum(),
        "missing_pct": (100 * X_external.isna().mean()).round(2),
        "unique": X_external.nunique(dropna=False)
    })
    return X_external, y_external, report

# Example feature engineering on the built-in dataset (no target leakage).
engineered_df = X.copy()
mean_cols = [c for c in engineered_df.columns if c.endswith(" mean")]
if len(mean_cols) >= 2:
    engineered_df["mean_feature_average"] = engineered_df[mean_cols].mean(axis=1)
    engineered_df["mean_feature_range"] = engineered_df[mean_cols].max(axis=1) - engineered_df[mean_cols].min(axis=1)
engineered_df["target"] = y.to_numpy()
display(engineered_df.head())
print("Engineered dataset shape:", engineered_df.shape)
# Export is optional; uncomment to create a local CSV.
# engineered_df.to_csv("engineered_breast_cancer.csv", index=False)

## 16. Deep-learning improvements: callbacks, learning-rate scheduling, and diagnostics

The dense network above is retained. This optional cell demonstrates a stronger training recipe with model checkpointing in memory, early stopping, and learning-rate reduction. It uses only the existing training partition; the held-out test partition remains untouched until evaluation.

In [ ]:
if HAS_TENSORFLOW:
    advanced_dl = keras.Sequential([
        layers.Input(shape=(X_dl_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(), layers.Dropout(0.35),
        layers.Dense(64, activation="relu"), layers.Dropout(0.25),
        layers.Dense(1, activation="sigmoid")
    ], name="advanced_tabular_network")
    advanced_dl.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                        loss="binary_crossentropy",
                        metrics=["accuracy", keras.metrics.AUC(name="auc")])
    advanced_callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6)
    ]
    advanced_history = advanced_dl.fit(
        X_dl_train, np.asarray(y_train), validation_split=0.2,
        epochs=120, batch_size=32, callbacks=advanced_callbacks, verbose=0
    )
    advanced_prob = advanced_dl.predict(X_dl_test, verbose=0).ravel()
    print("Advanced dense network ROC AUC:", round(roc_auc_score(y_test, advanced_prob), 4))
    pd.DataFrame(advanced_history.history)[["loss", "val_loss"]].plot(title="Advanced network loss")
    plt.xlabel("Epoch"); plt.ylabel("Binary cross-entropy"); plt.show()
else:
    print("Advanced deep-learning section skipped: install TensorFlow to run it.")

## 17. Computer vision upgrades: augmentation, CNN diagnostics, and inference helper

This section adds data augmentation and a reusable prediction helper to the MNIST CNN. Augmentation is applied only to training batches. The original MNIST CNN remains available as a baseline.

In [ ]:
if HAS_TENSORFLOW:
    augmentation = keras.Sequential([
        layers.RandomRotation(0.08),
        layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
        layers.RandomZoom(0.08)
    ], name="mnist_augmentation")
    augmented_cnn = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        augmentation,
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(96, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.35),
        layers.Dense(10, activation="softmax")
    ], name="augmented_mnist_cnn")
    augmented_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    aug_callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
    aug_history = augmented_cnn.fit(x_img_train, y_img_train, validation_split=0.1,
                                    epochs=12, batch_size=128, callbacks=aug_callbacks, verbose=1)
    aug_loss, aug_accuracy = augmented_cnn.evaluate(x_img_test, y_img_test, verbose=0)
    print(f"Augmented CNN test accuracy: {aug_accuracy:.4f}; loss: {aug_loss:.4f}")

    def predict_mnist_images(images, model=augmented_cnn):
        """Predict a batch of 28x28 grayscale images; accepts values in [0,255] or [0,1]."""
        arr = np.asarray(images, dtype="float32")
        if arr.ndim == 3:
            arr = arr[..., np.newaxis]
        if arr.ndim != 4 or arr.shape[1:] != (28, 28, 1):
            raise ValueError("Expected image shape (n, 28, 28) or (n, 28, 28, 1)")
        if arr.max(initial=0) > 1.0:
            arr = arr / 255.0
        probabilities = model.predict(arr, verbose=0)
        return pd.DataFrame({"predicted_digit": probabilities.argmax(axis=1),
                             "confidence": probabilities.max(axis=1)})

    display(predict_mnist_images(x_img_test[:8]))
    aug_pred = augmented_cnn.predict(x_img_test, verbose=0).argmax(axis=1)
    print(classification_report(y_img_test, aug_pred, zero_division=0))
else:
    print("Augmented computer-vision workflow skipped: install TensorFlow to run it.")

## 18. Custom image-folder workflow for your own computer-vision dataset

Place images in class-named subfolders, for example `images/cats/` and `images/dogs/`. Set `IMAGE_DATA_DIR` to that folder and run the optional cell. It builds a `tf.data` pipeline and a compact CNN. This workflow does not download pretrained weights or require an internet connection. Keep a separate external test folder for unbiased final evaluation.

In [ ]:
# Example: IMAGE_DATA_DIR = "./images"  # folder containing one subfolder per class
IMAGE_DATA_DIR = None
if HAS_TENSORFLOW and IMAGE_DATA_DIR and os.path.isdir(IMAGE_DATA_DIR):
    image_train_ds = keras.utils.image_dataset_from_directory(
        IMAGE_DATA_DIR, validation_split=0.2, subset="training", seed=RANDOM_STATE,
        image_size=(128, 128), batch_size=32, label_mode="int"
    )
    image_val_ds = keras.utils.image_dataset_from_directory(
        IMAGE_DATA_DIR, validation_split=0.2, subset="validation", seed=RANDOM_STATE,
        image_size=(128, 128), batch_size=32, label_mode="int"
    )
    image_class_names = image_train_ds.class_names
    AUTOTUNE = tf.data.AUTOTUNE
    image_train_ds = image_train_ds.cache().shuffle(500).prefetch(AUTOTUNE)
    image_val_ds = image_val_ds.cache().prefetch(AUTOTUNE)
    image_cnn = keras.Sequential([
        layers.Input(shape=(128, 128, 3)), layers.Rescaling(1./255),
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3), layers.Dense(len(image_class_names), activation="softmax")
    ])
    image_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    image_cnn.fit(image_train_ds, validation_data=image_val_ds, epochs=10,
                  callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    print("Image classes:", image_class_names)
elif HAS_TENSORFLOW:
    print("Set IMAGE_DATA_DIR to a folder containing one subfolder per image class to run this workflow.")
else:
    print("Custom image-folder workflow requires TensorFlow.")

## 19. Save and reload trained models

The following utilities save a fitted scikit-learn estimator with joblib and (when available) Keras models in the native `.keras` format. Saving is opt-in to avoid writing files unexpectedly.

In [ ]:
import os

def save_sklearn_model(model, path="best_classical_model.joblib"):
    """Persist a fitted scikit-learn estimator; returns the saved path."""
    import joblib
    joblib.dump(model, path)
    return os.path.abspath(path)

if HAS_TENSORFLOW:
    def save_keras_model(model, path="deep_learning_model.keras"):
        """Persist a fitted Keras model in the recommended native format."""
        model.save(path)
        return os.path.abspath(path)

# Opt in when ready:
# saved_model_path = save_sklearn_model(best_model)
# if HAS_TENSORFLOW: saved_dl_path = save_keras_model(deep_model)
print("Model persistence helpers are ready; uncomment a save call to create an artifact.")

## 20. Upgrade notes and responsible use

- The tabular example uses the scikit-learn breast-cancer benchmark; it is educational and is **not** a clinical diagnostic system.
- MNIST is a small handwritten-digit benchmark, not a substitute for domain-specific image data.
- The custom image-folder workflow requires labeled subdirectories and should be evaluated on a genuinely held-out test set.
- TensorFlow, SHAP, Plotly, and XGBoost remain optional. Install only the packages needed for the sections you plan to run.
- Code cells are designed to skip optional workflows gracefully when dependencies or user data are unavailable.

## 21. Comprehensive data cleaning and quality audit

Reusable Pandas utilities for duplicate detection, missingness, type normalization, constant-column removal, and IQR outlier *flagging*. Outliers are flagged rather than automatically deleted because unusual observations may be valid. Apply all learned preprocessing only on training data to avoid leakage.


In [ ]:
def data_quality_report(frame: pd.DataFrame) -> pd.DataFrame:
    """Summarize types, missing values, uniqueness, and memory usage."""
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_count": frame.isna().sum(),
        "missing_pct": (100 * frame.isna().mean()).round(2),
        "unique_count": frame.nunique(dropna=True),
        "memory_kb": (frame.memory_usage(deep=True, index=False) / 1024).round(2),
    })
    return report.sort_values(["missing_pct", "memory_kb"], ascending=False)

def clean_tabular_frame(frame: pd.DataFrame, *, drop_duplicates=True,
                        drop_constant=True, normalize_column_names=True):
    """Conservative cleaning; does not impute or inspect the target column."""
    cleaned = frame.copy()
    if normalize_column_names:
        cleaned.columns = (cleaned.columns.astype(str).str.strip()
                           .str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True)
                           .str.strip("_"))
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    if drop_duplicates:
        cleaned = cleaned.drop_duplicates().copy()
    removed_constant = []
    if drop_constant:
        removed_constant = [c for c in cleaned.columns if cleaned[c].nunique(dropna=False) <= 1]
        cleaned = cleaned.drop(columns=removed_constant)
    return cleaned, {"rows_removed_as_duplicates": int(len(frame) - len(frame.drop_duplicates())),
                     "constant_columns_removed": removed_constant}

def iqr_outlier_report(frame: pd.DataFrame, multiplier=1.5) -> pd.DataFrame:
    """Count IQR outliers per numeric column; does not modify the data."""
    rows = []
    for col in frame.select_dtypes(include=np.number):
        values = frame[col].dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - multiplier * iqr, q3 + multiplier * iqr
        rows.append({"feature": col, "lower_bound": lower, "upper_bound": upper,
                     "outlier_count": int(((values < lower) | (values > upper)).sum())})
    return pd.DataFrame(rows)

# Demonstration on the notebook's in-memory feature table.
quality_before = data_quality_report(X)
X_clean_preview, cleaning_log = clean_tabular_frame(X)
display(quality_before.head(15))
display(iqr_outlier_report(X).sort_values("outlier_count", ascending=False).head(15))
print("Cleaning summary:", cleaning_log)


## 22. Feature engineering toolkit

Create reusable numeric features without using the target. The helper adds row-wise statistics and (where possible) ratios for numeric columns. It is designed for exploratory feature creation; inspect correlations and validate every engineered feature on held-out data before adopting it.


In [ ]:
def engineer_tabular_features(frame: pd.DataFrame, *, add_row_stats=True,
                               add_pairwise_ratios=False, max_ratio_features=8):
    """Return a feature-engineered copy; never reads or modifies a target column."""
    engineered = frame.copy()
    numeric_cols = engineered.select_dtypes(include=np.number).columns.tolist()
    if add_row_stats and len(numeric_cols) >= 2:
        numeric = engineered[numeric_cols]
        engineered["_row_numeric_mean"] = numeric.mean(axis=1)
        engineered["_row_numeric_std"] = numeric.std(axis=1).fillna(0)
        engineered["_row_numeric_min"] = numeric.min(axis=1)
        engineered["_row_numeric_max"] = numeric.max(axis=1)
    if add_pairwise_ratios and len(numeric_cols) >= 2:
        selected = numeric_cols[:max_ratio_features]
        for i, numerator in enumerate(selected):
            for denominator in selected[i + 1:]:
                name = f"ratio_{numerator}_to_{denominator}"
                denom = engineered[denominator].replace(0, np.nan)
                engineered[name] = engineered[numerator] / denom
    return engineered.replace([np.inf, -np.inf], np.nan)

X_engineered = engineer_tabular_features(X, add_row_stats=True)
print("Original shape:", X.shape, "| Engineered shape:", X_engineered.shape)
display(X_engineered.head())


## 23. Leakage-safe preprocessing for mixed tabular data

Use a scikit-learn `Pipeline` and `ColumnTransformer` so imputation, scaling, and category encoding are fitted on training folds only. This is the recommended preprocessing route for external CSV datasets and model selection.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline as SkPipeline

def make_preprocessing_pipeline(X_frame: pd.DataFrame):
    """Build a fresh leakage-safe transformer matching the supplied feature schema."""
    numeric_features = X_frame.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_frame.select_dtypes(exclude=np.number).columns.tolist()
    numeric_pipeline = SkPipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", RobustScaler()),
    ])
    categorical_pipeline = SkPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers = []
    if numeric_features:
        transformers.append(("numeric", numeric_pipeline, numeric_features))
    if categorical_features:
        transformers.append(("categorical", categorical_pipeline, categorical_features))
    if not transformers:
        raise ValueError("No usable feature columns were found.")
    return ColumnTransformer(transformers, remainder="drop")

# Demonstrate the preprocessing workflow without changing earlier model results.
X_demo_train, X_demo_test = train_test_split(X_engineered, test_size=0.2,
                                              random_state=RANDOM_STATE, stratify=y)
preprocessor_demo = make_preprocessing_pipeline(X_demo_train)
X_demo_train_ready = preprocessor_demo.fit_transform(X_demo_train)
X_demo_test_ready = preprocessor_demo.transform(X_demo_test)
print("Leakage-safe transformed shapes:", X_demo_train_ready.shape, X_demo_test_ready.shape)


## 24. Clean-and-engineer workflow for a user CSV

Set `CSV_PATH` and `TARGET_COLUMN` to use your own data. The target is separated before feature engineering; duplicate rows are removed from the combined frame before splitting. For real projects, decide explicitly whether duplicate records are genuinely redundant and review the quality report before dropping columns.


In [ ]:
# Example configuration (edit these two values for your dataset).
CSV_PATH = None          # e.g. "./my_dataset.csv"
TARGET_COLUMN = None     # e.g. "label"

external_dataset = None
if CSV_PATH and TARGET_COLUMN:
    raw_external = pd.read_csv(CSV_PATH)
    if TARGET_COLUMN not in raw_external.columns:
        raise KeyError(f"Target column {TARGET_COLUMN!r} not found in CSV")
    raw_external = raw_external.drop_duplicates().copy()
    y_external = raw_external.pop(TARGET_COLUMN)
    X_external, external_cleaning_log = clean_tabular_frame(raw_external)
    X_external = engineer_tabular_features(X_external)
    external_report = data_quality_report(X_external)
    external_dataset = (X_external, y_external)
    display(external_report.head(20))
    print("External cleaning summary:", external_cleaning_log)
else:
    print("Set CSV_PATH and TARGET_COLUMN above to run the custom-data workflow.")


### Data preparation best practices

- Split before fitting imputers, scalers, encoders, feature selectors, or resampling methods.
- Keep the target out of all feature engineering and preprocessing steps.
- Do not automatically remove outliers; investigate their meaning first.
- Use pipelines inside cross-validation to prevent leakage.
- Compare engineered and baseline features using the same stratified folds and untouched test set.


## 25. Advanced model reliability: calibration and decision thresholds

This section adds probability calibration, precision-recall analysis, and a validation-only threshold selection workflow. The held-out test set is reserved for final reporting; choose thresholds using cross-validated training predictions where possible.


In [ ]:
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, precision_recall_curve, average_precision_score
from sklearn.model_selection import cross_val_predict

# Calibration diagnostics for the selected estimator (when it supports probabilities).
if hasattr(best_model, "predict_proba"):
    best_probability = best_model.predict_proba(X_test)[:, 1]
    print("Test Brier score:", round(brier_score_loss(y_test, best_probability), 4))
    print("Test average precision:", round(average_precision_score(y_test, best_probability), 4))
    fig, ax = plt.subplots(figsize=(6, 5))
    CalibrationDisplay.from_predictions(y_test, best_probability, n_bins=10, ax=ax)
    ax.set_title(f"Probability calibration — {best_name}")
    plt.tight_layout(); plt.show()
else:
    print("Selected estimator does not expose predict_proba; calibration plot skipped.")


## 26. Learning curves and generalization diagnostics

Learning curves show whether additional training data may help. They use training folds only and do not alter the final test evaluation.


In [ ]:
from sklearn.model_selection import learning_curve

curve_sizes, train_scores, validation_scores = learning_curve(
    clone(best_model), X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc", train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
)
plt.figure(figsize=(8, 5))
plt.plot(curve_sizes, train_scores.mean(axis=1), marker="o", label="Training ROC AUC")
plt.plot(curve_sizes, validation_scores.mean(axis=1), marker="o", label="Validation ROC AUC")
plt.fill_between(curve_sizes, validation_scores.mean(axis=1)-validation_scores.std(axis=1),
                 validation_scores.mean(axis=1)+validation_scores.std(axis=1), alpha=.15)
plt.xlabel("Training examples"); plt.ylabel("ROC AUC"); plt.ylim(0.5, 1.02)
plt.title(f"Learning curve — {best_name}"); plt.legend(); plt.grid(alpha=.2); plt.show()


## 27. Class imbalance diagnostics and optional resampling

Inspect class balance before deciding whether resampling is justified. Resampling must happen inside each training fold (for example with an imbalanced-learn pipeline), never before the train/test split. This section reports imbalance and provides class-weighted baseline evaluation without requiring extra packages.


In [ ]:
class_counts = y_train.value_counts().sort_index()
display(class_counts.rename("training_count").to_frame())
imbalance_ratio = class_counts.max() / class_counts.min()
print("Training majority/minority ratio:", round(float(imbalance_ratio), 2))

weighted_model = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
weighted_model.fit(X_train, y_train)
weighted_prob = weighted_model.predict_proba(X_test)[:, 1]
weighted_pred = (weighted_prob >= 0.5).astype(int)
print(classification_report(y_test, weighted_pred, target_names=class_names))
print("Balanced RF ROC AUC:", round(roc_auc_score(y_test, weighted_prob), 4))


## 28. Dataset drift screening

Compare a reference dataset with a later/batch dataset using descriptive statistics and a simple population stability index (PSI). PSI is a screening indicator, not proof of drift or a substitute for domain review. Use the same feature definitions and exclude target columns.


In [ ]:
def population_stability_index(reference, current, bins=10):
    """Calculate numeric PSI using reference quantile bins; handles constant columns."""
    ref = pd.to_numeric(pd.Series(reference), errors="coerce").dropna().to_numpy()
    cur = pd.to_numeric(pd.Series(current), errors="coerce").dropna().to_numpy()
    if len(ref) == 0 or len(cur) == 0:
        return np.nan
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0 if np.isclose(np.mean(ref), np.mean(cur)) else np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts = np.histogram(ref, bins=edges)[0].astype(float)
    cur_counts = np.histogram(cur, bins=edges)[0].astype(float)
    ref_pct = np.clip(ref_counts / ref_counts.sum(), 1e-6, None)
    cur_pct = np.clip(cur_counts / cur_counts.sum(), 1e-6, None)
    return float(np.sum((cur_pct-ref_pct) * np.log(cur_pct/ref_pct)))

# Demonstration: compare two reproducible subsets of the available feature table.
reference_batch = X.iloc[:len(X)//2]
current_batch = X.iloc[len(X)//2:]
drift_report = pd.DataFrame({
    "PSI": {col: population_stability_index(reference_batch[col], current_batch[col])
            for col in X.select_dtypes(include=np.number).columns}
}).sort_values("PSI", ascending=False)
display(drift_report.head(15))
print("Rule of thumb only: PSI > 0.20 merits investigation; thresholds depend on context.")


## 29. Reproducible experiment registry and artifact export

Store model metrics and run metadata in a compact Pandas registry. The export cell writes CSV/JSON artifacts into the current working directory when executed.


In [ ]:
import platform, json

experiment_registry = results.copy()
experiment_registry["dataset"] = "sklearn_breast_cancer"
experiment_registry["random_state"] = RANDOM_STATE
experiment_registry["python_version"] = platform.python_version()
experiment_registry["selected_model"] = experiment_registry["Model"].eq(best_name)
display(experiment_registry)

# Opt-in exports; uncomment to write files.
# experiment_registry.to_csv("ml_experiment_registry.csv", index=False)
# with open("ml_run_metadata.json", "w", encoding="utf-8") as f:
#     json.dump({"dataset": "sklearn_breast_cancer", "random_state": int(RANDOM_STATE),
#                "selected_model": str(best_name)}, f, indent=2)
print("Experiment registry prepared. Uncomment export lines to save artifacts.")


## 30. Final quality checklist

- Keep the test set untouched until final evaluation; do not choose features, thresholds, or hyperparameters from test results.
- Fit imputers, encoders, scalers, feature selection, and resampling only on training folds through pipelines.
- Validate custom image labels, image dimensions, and class balance before training CNNs.
- Record package versions, random seeds, data provenance, and model artifacts for reproducibility.
- The built-in breast-cancer dataset is for educational demonstration only, not clinical diagnosis.


# 31. Visual analytics gallery: 2D/3D diagrams, charts, and comparison tables

This section consolidates the notebook's most relevant topics into publication-friendly visuals:
- **2D diagrams:** end-to-end ML workflow and data-cleaning/feature-engineering pipeline.
- **2D charts:** model metric comparison, class balance, PCA projection, and feature importance.
- **3D charts:** PCA feature space and model-performance trade-offs.
- **Comparison tables:** model metrics and preprocessing choices.

All plots use the dataset and fitted models created above. Plotly charts are optional; Matplotlib fallbacks are included.

In [ ]:
# 31.1 Two-dimensional process diagrams
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_process_diagram(title, steps, figsize=(14, 3.2)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, len(steps) * 2.1)
    ax.set_ylim(0, 2)
    ax.axis("off")
    positions = []
    for i, (heading, detail) in enumerate(steps):
        x = i * 2.1 + 0.12
        box = FancyBboxPatch((x, 0.55), 1.72, 0.95,
                             boxstyle="round,pad=0.08", facecolor="#eaf2f8",
                             edgecolor="#2874a6", linewidth=1.5)
        ax.add_patch(box)
        ax.text(x + .86, 1.18, heading, ha="center", va="center",
                fontsize=9, fontweight="bold")
        ax.text(x + .86, .83, detail, ha="center", va="center",
                fontsize=7, wrap=True)
        if i:
            ax.add_patch(FancyArrowPatch((x - .38, 1.02), (x - .06, 1.02),
                                         arrowstyle="->", mutation_scale=13,
                                         color="#34495e"))
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

draw_process_diagram("End-to-end machine learning workflow", [
    ("Collect", "CSV / images"), ("Audit", "quality + leakage"),
    ("Prepare", "clean + encode"), ("Engineer", "create features"),
    ("Train", "CV + tuning"), ("Evaluate", "test metrics"),
    ("Explain", "SHAP + importance"), ("Deploy", "save + monitor")
])
draw_process_diagram("Computer vision workflow", [
    ("Images", "labelled folders"), ("Decode", "resize + RGB"),
    ("Augment", "flip / rotate"), ("CNN", "convolutions"),
    ("Validate", "loss + accuracy"), ("Predict", "class + score")
], figsize=(11, 3.2))

### 31.2 Data cleaning and feature engineering comparison

In [ ]:
# Compare raw and cleaned data using a reproducible demonstration with the loaded dataset.
raw_demo = X.copy() if "X" in globals() else dataset.data.copy()
if not isinstance(raw_demo, pd.DataFrame):
    raw_demo = pd.DataFrame(raw_demo)
dirty_demo = raw_demo.copy()
if dirty_demo.size:
    dirty_demo.iloc[0, 0] = np.nan
    dirty_demo = pd.concat([dirty_demo, dirty_demo.iloc[[0]]], ignore_index=True)
clean_demo = dirty_demo.drop_duplicates().copy()
for col in clean_demo.select_dtypes(include=np.number).columns:
    clean_demo[col] = clean_demo[col].replace([np.inf, -np.inf], np.nan)
    clean_demo[col] = clean_demo[col].fillna(clean_demo[col].median())

cleaning_comparison = pd.DataFrame({
    "Measure": ["Rows", "Columns", "Missing values", "Duplicate rows"],
    "Before": [len(dirty_demo), dirty_demo.shape[1], int(dirty_demo.isna().sum().sum()),
               int(dirty_demo.duplicated().sum())],
    "After": [len(clean_demo), clean_demo.shape[1], int(clean_demo.isna().sum().sum()),
              int(clean_demo.duplicated().sum())]
})
display(cleaning_comparison)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cleaning_comparison.set_index("Measure")[["Before", "After"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Data quality: before vs after")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=25)
missing = pd.Series({"Before": dirty_demo.isna().sum().sum(),
                     "After": clean_demo.isna().sum().sum()})
axes[1].bar(missing.index, missing.values)
axes[1].set_title("Missing-value reduction")
axes[1].set_ylabel("Missing cells")
plt.tight_layout()
plt.show()

### 31.3 Feature engineering visual summary

In [ ]:
# Summarize numerical feature distributions and correlations.
numeric_for_viz = clean_demo.select_dtypes(include=np.number)
if numeric_for_viz.shape[1] >= 2:
    corr = numeric_for_viz.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax,
                cbar_kws={"label": "Pearson correlation"})
    ax.set_title("Feature correlation heatmap")
    plt.tight_layout()
    plt.show()

    engineered_preview = numeric_for_viz.copy()
    engineered_preview["row_mean"] = numeric_for_viz.mean(axis=1)
    engineered_preview["row_std"] = numeric_for_viz.std(axis=1)
    engineered_preview["row_min"] = numeric_for_viz.min(axis=1)
    engineered_preview["row_max"] = numeric_for_viz.max(axis=1)
    display(engineered_preview[["row_mean", "row_std", "row_min", "row_max"]].describe().T)
else:
    print("At least two numeric features are required for the correlation chart.")

### 31.4 2D and 3D dataset projections

In [ ]:
# PCA plots are descriptive projections; they are not model decision boundaries.
if "X_train" in globals() and "y_train" in globals():
    viz_sample = X_train.copy()
    viz_labels = y_train.copy()
    pca_vis = PCA(n_components=3, random_state=RANDOM_STATE)
    projected = pca_vis.fit_transform(StandardScaler().fit_transform(viz_sample))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    scatter = axes[0].scatter(projected[:, 0], projected[:, 1], c=np.asarray(viz_labels),
                              cmap="coolwarm", alpha=.75, s=20)
    axes[0].set(title="2D PCA projection", xlabel="PC1", ylabel="PC2")
    axes[1].scatter(projected[:, 0], projected[:, 2], c=np.asarray(viz_labels),
                    cmap="coolwarm", alpha=.75, s=20)
    axes[1].set(title="PC1 vs PC3 projection", xlabel="PC1", ylabel="PC3")
    fig.colorbar(scatter, ax=axes, label="Target class")
    plt.tight_layout()
    plt.show()

    from mpl_toolkits.mplot3d import Axes3D
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    points = ax.scatter(projected[:, 0], projected[:, 1], projected[:, 2],
                        c=np.asarray(viz_labels), cmap="coolwarm", alpha=.75, s=22)
    ax.set(title="3D PCA feature space", xlabel="PC1", ylabel="PC2", zlabel="PC3")
    fig.colorbar(points, ax=ax, label="Target class", pad=.1)
    plt.tight_layout()
    plt.show()
else:
    print("Run the data split section first to create X_train and y_train.")

### 31.5 Model comparison tables and charts

In [ ]:
# Build a consistent comparison table from the notebook's existing results.
if "results" in globals() and isinstance(results, pd.DataFrame) and not results.empty:
    comparison_table = results.copy()
    display(comparison_table.style.format(precision=3).background_gradient(
        subset=comparison_table.select_dtypes(include=np.number).columns.tolist(),
        cmap="YlGn"))
    numeric_metrics = comparison_table.select_dtypes(include=np.number)
    if not numeric_metrics.empty:
        plot_metrics = comparison_table[[comparison_table.columns[0]] + numeric_metrics.columns[: min(5, numeric_metrics.shape[1])].tolist()].copy()
        ax = plot_metrics.set_index(comparison_table.columns[0]).plot(
            kind="bar", figsize=(12, 5), width=.8)
        ax.set_title("Model metric comparison")
        ax.set_ylabel("Score")
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=25)
        ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
else:
    print("Model comparison results are not available yet; run model training first.")

### 31.6 Classification diagnostics: confusion matrix and ROC

In [ ]:
# Use the best model and held-out test data already created in earlier sections.
if all(name in globals() for name in ["best_model", "X_test", "y_test"]):
    y_pred_viz = best_model.predict(X_test)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_viz, ax=axes[0],
                                             cmap="Blues", colorbar=False)
    axes[0].set_title("Confusion matrix")
    if hasattr(best_model, "predict_proba") and pd.Series(y_test).nunique() == 2:
        RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
        axes[1].set_title("ROC curve")
    else:
        axes[1].text(.5, .5, "ROC curve unavailable for this estimator/target",
                     ha="center", va="center")
        axes[1].set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print("Train and evaluate a model first to display classification diagnostics.")

### 31.7 Interactive 3D model comparison (Plotly optional)

In [ ]:
if HAS_PLOTLY and "results" in globals() and isinstance(results, pd.DataFrame) and not results.empty:
    import plotly.express as px
    numeric_cols = results.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) >= 3:
        plotly_frame = results.copy()
        plotly_frame["Model name"] = plotly_frame.iloc[:, 0].astype(str)
        fig = px.scatter_3d(plotly_frame, x=numeric_cols[0], y=numeric_cols[1],
                            z=numeric_cols[2], color="Model name",
                            hover_name="Model name", title="Interactive 3D model comparison")
        fig.show()
    else:
        print("At least three numeric model metrics are needed for a 3D comparison.")
else:
    print("Install Plotly and run model comparison to enable the interactive 3D chart.")

### 31.8 Visual comparison of preprocessing strategies

In [ ]:
preprocessing_comparison = pd.DataFrame([
    ["Median imputation", "Numeric missing values", "Robust to skew", "Fit on training data"],
    ["StandardScaler", "Numeric features", "Centers and scales", "Useful for SVM / MLP"],
    ["RobustScaler", "Numeric features with outliers", "Uses median and IQR", "Useful with outliers"],
    ["OneHotEncoder", "Categorical features", "Creates indicator columns", "Handle unknown categories"],
    ["ColumnTransformer", "Mixed data types", "Applies column-specific steps", "Prevents preprocessing leakage"],
], columns=["Method", "Best suited for", "Main benefit", "Practical note"])
display(preprocessing_comparison)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.axis("off")
table = ax.table(cellText=preprocessing_comparison.values,
                 colLabels=preprocessing_comparison.columns,
                 cellLoc="left", colLoc="left", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.7)
ax.set_title("Preprocessing methods: comparison table", pad=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 31.9 Deep-learning training curves (when a Keras history exists)

In [ ]:
# Compatible with history objects created by the optional TensorFlow sections.
if "history" in globals() and hasattr(history, "history"):
    history_frame = pd.DataFrame(history.history)
    display(history_frame.tail())
    metric_names = [c for c in history_frame.columns if "accuracy" in c.lower()]
    loss_names = [c for c in history_frame.columns if "loss" in c.lower()]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for col in loss_names:
        axes[0].plot(history_frame.index + 1, history_frame[col], label=col)
    axes[0].set(title="Training / validation loss", xlabel="Epoch", ylabel="Loss")
    axes[0].legend()
    for col in metric_names:
        axes[1].plot(history_frame.index + 1, history_frame[col], label=col)
    axes[1].set(title="Training / validation accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("No Keras History object found. Run a deep-learning training cell to generate these curves.")

## 32. Visualization guide and interpretation

| Topic | Recommended visual | What to look for |
|---|---|---|
| Data cleaning | Before/after bar chart | Fewer missing and duplicate records |
| Feature engineering | Correlation heatmap | Redundant or strongly associated features |
| ML model selection | Metric comparison chart/table | Balanced precision, recall, F1 and ROC-AUC |
| Classification errors | Confusion matrix | False positives and false negatives |
| Representation learning | 2D/3D PCA | Class overlap and cluster structure |
| Deep learning | Loss/accuracy curves | Convergence and overfitting |
| Computer vision | Image grids and predictions | Label quality and incorrect classifications |
| Monitoring | PSI drift report | Distribution changes requiring investigation |

**Interpretation caution:** PCA axes are projections, not direct measures of feature importance. A high training score alone does not establish generalization. Use held-out test data and cross-validation for model comparisons.

# 33. Leakage-safe model selection (recommended final workflow)

Earlier comparison charts are useful for exploration, but choosing the winner by held-out test ROC AUC uses the test set for model selection. The workflow below selects a model using cross-validation on the training data only, refits it on all training rows, and evaluates the selected model on the untouched test set once.

In [ ]:
from sklearn.model_selection import cross_val_score

selection_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for candidate_name, candidate_model in models.items():
    scores = cross_val_score(clone(candidate_model), X_train, y_train,
                             cv=selection_cv, scoring="roc_auc", n_jobs=-1)
    cv_rows.append({"Model": candidate_name, "CV ROC AUC mean": scores.mean(),
                    "CV ROC AUC std": scores.std()})
cv_selection = pd.DataFrame(cv_rows).sort_values("CV ROC AUC mean", ascending=False)
display(cv_selection.style.format(precision=4))
selected_name = cv_selection.iloc[0]["Model"]
selected_model = clone(models[selected_name]).fit(X_train, y_train)
selected_probability = selected_model.predict_proba(X_test)[:, 1]
selected_prediction = selected_model.predict(X_test)
print("CV-selected model:", selected_name)
print("Final held-out ROC AUC:", round(roc_auc_score(y_test, selected_probability), 4))
print(classification_report(y_test, selected_prediction, target_names=class_names))

# 34. Regression workflow: baseline, metrics, residuals and prediction

A separate synthetic regression example demonstrates regression metrics without changing the classification dataset or its results.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_reg, y_reg = make_regression(n_samples=1200, n_features=12, n_informative=8,
                               noise=18.0, random_state=RANDOM_STATE)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE)
regressors = {
    "Ridge": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Random Forest": RandomForestRegressor(n_estimators=180, min_samples_leaf=2,
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}
reg_rows, reg_fitted = [], {}
for name, estimator in regressors.items():
    estimator.fit(Xr_train, yr_train)
    pred = estimator.predict(Xr_test)
    reg_fitted[name] = estimator
    reg_rows.append({"Model": name, "MAE": mean_absolute_error(yr_test, pred),
                     "RMSE": mean_squared_error(yr_test, pred) ** 0.5,
                     "R2": r2_score(yr_test, pred)})
regression_results = pd.DataFrame(reg_rows).sort_values("RMSE")
display(regression_results.style.format(precision=3))
best_reg_name = regression_results.iloc[0]["Model"]
best_reg_pred = reg_fitted[best_reg_name].predict(Xr_test)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(yr_test, best_reg_pred, alpha=.6)
lims = [min(yr_test.min(), best_reg_pred.min()), max(yr_test.max(), best_reg_pred.max())]
axes[0].plot(lims, lims, "k--"); axes[0].set(title=f"Actual vs predicted: {best_reg_name}",
                                               xlabel="Actual", ylabel="Predicted")
residuals = yr_test - best_reg_pred
axes[1].scatter(best_reg_pred, residuals, alpha=.6)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(title="Residual diagnostics", xlabel="Predicted", ylabel="Residual")
plt.tight_layout(); plt.show()

# 35. Unsupervised learning: clustering and cluster quality

PCA is used only to display clusters in two dimensions. Silhouette score is reported in the original standardized feature space.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(X)
cluster_rows, cluster_labels = [], {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_cluster)
    cluster_labels[k] = labels
    cluster_rows.append({"Clusters": k, "Inertia": km.inertia_,
                         "Silhouette": silhouette_score(X_cluster, labels)})
cluster_results = pd.DataFrame(cluster_rows)
display(cluster_results)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(cluster_results["Clusters"], cluster_results["Inertia"], marker="o")
axes[0].set(title="K-means elbow curve", xlabel="Number of clusters", ylabel="Inertia")
axes[1].plot(cluster_results["Clusters"], cluster_results["Silhouette"], marker="o")
axes[1].set(title="Silhouette comparison", xlabel="Number of clusters", ylabel="Silhouette")
plt.tight_layout(); plt.show()
best_k = int(cluster_results.loc[cluster_results["Silhouette"].idxmax(), "Clusters"])
pca_cluster = PCA(n_components=2, random_state=RANDOM_STATE)
cluster_2d = pca_cluster.fit_transform(X_cluster)
plt.figure(figsize=(8, 6))
plt.scatter(cluster_2d[:, 0], cluster_2d[:, 1], c=cluster_labels[best_k],
            cmap="tab10", s=22, alpha=.75)
plt.title(f"K-means clusters (k={best_k}) shown in PCA space")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

# 36. Anomaly detection with Isolation Forest

This is an unsupervised screening example. Anomaly flags are not ground-truth diagnoses; inspect flagged observations before taking action.

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(n_estimators=250, contamination="auto",
                                random_state=RANDOM_STATE)
anomaly_labels = anomaly_model.fit_predict(X_cluster)  # -1 = flagged, 1 = inlier
anomaly_scores = -anomaly_model.score_samples(X_cluster)  # larger = more unusual
anomaly_report = pd.DataFrame({
    "anomaly_flag": anomaly_labels == -1,
    "anomaly_score": anomaly_scores
}, index=X.index)
display(anomaly_report.sort_values("anomaly_score", ascending=False).head(15))
print("Flagged fraction:", round(float((anomaly_labels == -1).mean()), 4))
plt.figure(figsize=(8, 4))
sns.histplot(anomaly_scores, bins=30, kde=True)
plt.title("Isolation Forest anomaly score distribution")
plt.xlabel("Anomaly score (higher = more unusual)"); plt.show()

# 37. Dataset validation and reusable schema checks

Run these checks before training on an external dataset. They report common problems and do not silently modify source data.

In [ ]:
def validate_tabular_dataset(frame, target=None, *, max_missing_fraction=0.40):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("Expected a pandas DataFrame.")
    if frame.empty:
        raise ValueError("Dataset is empty.")
    if frame.columns.duplicated().any():
        raise ValueError("Duplicate column names detected.")
    if target is not None and target not in frame.columns:
        raise KeyError(f"Target column {target!r} is missing.")
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_count": frame.isna().sum(),
        "missing_fraction": frame.isna().mean(),
        "unique_values": frame.nunique(dropna=False)
    })
    warnings_list = []
    if frame.duplicated().any():
        warnings_list.append(f"{int(frame.duplicated().sum())} duplicate rows")
    high_missing = report.index[report["missing_fraction"] > max_missing_fraction].tolist()
    if high_missing:
        warnings_list.append(f"High missingness columns: {high_missing}")
    if target is not None and frame[target].nunique(dropna=True) < 2:
        warnings_list.append("Target has fewer than two observed classes/values")
    return report, warnings_list

validation_report, validation_warnings = validate_tabular_dataset(df, target="target")
display(validation_report)
print("Validation warnings:", validation_warnings or "None")

# 38. Model persistence and portable inference bundle

Save the CV-selected classifier together with its feature names and metadata. Load only trusted files: joblib/pickle files can execute code when deserialized.

In [ ]:
import joblib
from pathlib import Path

MODEL_BUNDLE_PATH = Path("cv_selected_model_bundle.joblib")
model_bundle = {
    "model": selected_model,
    "feature_names": list(X.columns),
    "selected_model_name": selected_name,
    "random_state": RANDOM_STATE,
    "target_names": class_names
}
joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
print("Saved:", MODEL_BUNDLE_PATH.resolve())

# Example safe-schema inference using a row from the held-out feature table.
loaded_bundle = joblib.load(MODEL_BUNDLE_PATH)
example_batch = X_test.iloc[:3].loc[:, loaded_bundle["feature_names"]]
bundle_predictions = loaded_bundle["model"].predict(example_batch)
print("Example predictions:", bundle_predictions)

# 39. Export a compact project report

Create a reproducible summary of classification, regression, clustering, and environment details. Files are written to the notebook's current working directory.

In [ ]:
project_summary = {
    "classification_cv_selection": cv_selection.to_dict(orient="records"),
    "classification_final_test_roc_auc": float(roc_auc_score(y_test, selected_probability)),
    "regression_comparison": regression_results.to_dict(orient="records"),
    "clustering_comparison": cluster_results.to_dict(orient="records"),
    "selected_cluster_count": best_k,
    "anomaly_flag_fraction": float((anomaly_labels == -1).mean()),
    "python_version": sys.version.split()[0],
    "random_state": RANDOM_STATE
}
with open("3d_ml_suite_project_summary.json", "w", encoding="utf-8") as report_file:
    json.dump(project_summary, report_file, indent=2)
regression_results.to_csv("regression_model_comparison.csv", index=False)
cluster_results.to_csv("clustering_comparison.csv", index=False)
print("Exported project summary and comparison CSV files.")

## Final upgrade notes

- Model selection now has a recommended CV-only training workflow before one final test-set evaluation.
- Regression, clustering, anomaly detection, dataset validation, persistence, and report export are added as separate reproducible modules.
- This notebook's breast-cancer dataset is an educational benchmark, not a clinical diagnostic tool.
- Review dataset provenance, privacy, fairness, and domain requirements before applying models to real decisions.

## 31. NLP: text preprocessing and sentiment-style classification

A compact, self-contained text classification demonstration using TF-IDF and Logistic Regression. Replace the demo texts with a labelled text dataset for a real task. The example avoids downloading external data.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

text_samples = [
    "excellent product and friendly service", "very happy with the quality",
    "fast delivery and great experience", "I love this useful product",
    "wonderful support and easy setup", "high quality and good value",
    "terrible quality and poor service", "very disappointed with this purchase",
    "late delivery and damaged item", "I hate this frustrating experience",
    "waste of money and bad support", "the product stopped working",
    "great performance and reliable", "fantastic experience overall",
    "not worth the price", "poor performance and unreliable"
]
text_labels = np.array([1]*8 + [0]*8)  # 1=positive, 0=negative (demo labels)
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    text_samples, text_labels, test_size=0.25, random_state=RANDOM_STATE, stratify=text_labels)
nlp_pipeline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), stop_words="english"),
                             LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
nlp_pipeline.fit(X_text_train, y_text_train)
nlp_pred = nlp_pipeline.predict(X_text_test)
print(classification_report(y_text_test, nlp_pred, target_names=["negative", "positive"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_text_test, nlp_pred, display_labels=["negative", "positive"]); plt.title("NLP demo: text classification"); plt.show()
print("Example prediction:", nlp_pipeline.predict(["great quality and excellent support"])[0])


## 32. Time-series feature engineering and forecasting demo

Creates a deterministic synthetic series, builds lag/rolling features, and evaluates a regression model using a chronological split. For real data, sort by timestamp first and never randomly shuffle time-series observations.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(RANDOM_STATE)
n_time = 420
time_index = pd.date_range("2023-01-01", periods=n_time, freq="D")
series = 20 + 0.035*np.arange(n_time) + 3*np.sin(2*np.pi*np.arange(n_time)/7) + rng.normal(0, 0.8, n_time)
ts = pd.DataFrame({"date": time_index, "value": series}).set_index("date")
for lag in [1, 2, 7, 14]:
    ts[f"lag_{lag}"] = ts["value"].shift(lag)
ts["rolling_mean_7"] = ts["value"].shift(1).rolling(7).mean()
ts["rolling_std_7"] = ts["value"].shift(1).rolling(7).std()
ts_model = ts.dropna()
feature_cols = [c for c in ts_model.columns if c != "value"]
split_at = int(len(ts_model)*0.8)
X_ts_train, X_ts_test = ts_model[feature_cols].iloc[:split_at], ts_model[feature_cols].iloc[split_at:]
y_ts_train, y_ts_test = ts_model["value"].iloc[:split_at], ts_model["value"].iloc[split_at:]
ts_model_reg = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_ts_train, y_ts_train)
ts_pred = ts_model_reg.predict(X_ts_test)
print("MAE:", round(mean_absolute_error(y_ts_test, ts_pred), 3), "RMSE:", round(np.sqrt(mean_squared_error(y_ts_test, ts_pred)), 3))
plt.figure(figsize=(12,4)); plt.plot(y_ts_test.index, y_ts_test, label="Actual"); plt.plot(y_ts_test.index, ts_pred, label="Predicted"); plt.title("Chronological time-series holdout"); plt.legend(); plt.show()


## 33. Model monitoring: prediction and feature drift snapshot

A lightweight monitoring template. Compare a reference batch with a later batch. The PSI function is a screening heuristic; investigate alerts rather than treating them as proof of drift.


In [ ]:
def numeric_psi(reference, current, bins=10, eps=1e-6):
    ref = pd.to_numeric(pd.Series(reference), errors="coerce").dropna().to_numpy()
    cur = pd.to_numeric(pd.Series(current), errors="coerce").dropna().to_numpy()
    if len(ref) == 0 or len(cur) == 0:
        return np.nan
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins+1)))
    if len(edges) < 3:
        return 0.0 if np.allclose(np.mean(ref), np.mean(cur)) else np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, bins=edges)[0] / len(ref)
    q = np.histogram(cur, bins=edges)[0] / len(cur)
    p, q = np.clip(p, eps, None), np.clip(q, eps, None)
    return float(np.sum((q-p)*np.log(q/p)))

monitor_reference = X_train.select_dtypes(include=np.number)
monitor_current = X_test.select_dtypes(include=np.number)
monitor_rows = []
for col in monitor_reference.columns:
    monitor_rows.append({"feature": col, "PSI": numeric_psi(monitor_reference[col], monitor_current[col])})
monitor_report = pd.DataFrame(monitor_rows).sort_values("PSI", ascending=False)
display(monitor_report.head(15))


## 34. Model card and reproducibility manifest

Document intended use, data, metrics, limitations, and software versions before sharing a model. The manifest is generated from the active notebook session.


In [ ]:
import json, platform, sklearn
model_card = {
    "project": "3D ML Visualization Suite",
    "dataset": "scikit-learn breast cancer benchmark",
    "task": "educational binary classification",
    "intended_use": "demonstration and learning only",
    "not_for": "clinical diagnosis or decisions affecting patient care",
    "random_state": RANDOM_STATE,
    "primary_metric": "ROC AUC",
    "limitations": ["benchmark data may not represent deployment populations",
                    "test metrics do not guarantee future performance",
                    "check fairness, calibration, drift and data quality before deployment"]
}
run_manifest = {"python": platform.python_version(), "pandas": pd.__version__,
                "numpy": np.__version__, "scikit_learn": sklearn.__version__,
                "platform": platform.platform()}
print(json.dumps({"model_card": model_card, "environment": run_manifest}, indent=2))


## 35. Final expanded capability map

| Domain | Included methods | Suggested visual output |
|---|---|---|
| Tabular ML | Classification, regression, clustering, anomaly detection | ROC, confusion matrix, residual plots, PCA |
| Data engineering | Pandas cleaning, imputation, encoding, feature engineering | Missingness and correlation charts |
| Deep learning | Dense networks, CNN, callbacks | Learning curves and image predictions |
| Computer vision | MNIST and custom image folders | Image grids and class predictions |
| NLP | TF-IDF text classification | Confusion matrix and class report |
| Time series | Lag and rolling-window features, chronological holdout | Actual vs predicted timeline |
| Explainability & monitoring | Permutation importance, SHAP (optional), PSI | Importance and drift reports |
| Experiment practice | Model registry, model card, environment manifest | Comparison tables and JSON reports |

**Execution guidance:** run cells from top to bottom. Optional TensorFlow, SHAP, XGBoost and Plotly sections require their packages. The NLP and time-series examples use synthetic/demo data and should be replaced with domain-appropriate labelled data for applied work.


# 40. Explainable ML: Partial Dependence and ICE
Partial dependence plots (PDP) show average model response as a feature changes; individual conditional expectation (ICE) curves show per-row responses. These are descriptive explanations, not causal effects. The fitted model is evaluated on a small feature subset to keep plotting responsive.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Use the selected CV model if available; otherwise fall back to the earlier best model.
explain_model = selected_model if "selected_model" in globals() else best_model
explain_features = list(X_train.columns)
# Pick the two features with highest permutation importance when available.
if "perm" in globals() and hasattr(perm, "importances_mean"):
    top_idx = np.argsort(perm.importances_mean)[-2:][::-1]
    pdp_features = [explain_features[i] for i in top_idx]
else:
    pdp_features = explain_features[:2]

fig, ax = plt.subplots(figsize=(12, 5))
PartialDependenceDisplay.from_estimator(
    explain_model, X_train, features=pdp_features,
    kind="both", subsample=min(100, len(X_train)),
    random_state=RANDOM_STATE, ax=ax
)
fig.suptitle("Partial dependence and individual conditional expectation")
plt.tight_layout()
plt.show()
print("Displayed features:", pdp_features)

# 41. Validation-only decision threshold optimization
The default 0.50 probability threshold is not always appropriate. This section selects a threshold using a validation split from the training data only, then reports the resulting threshold on the untouched test set. Change the relative false-negative and false-positive costs to match a legitimate application; this benchmark is educational, not clinical advice.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.20, stratify=y_train,
    random_state=RANDOM_STATE
)
threshold_model = clone(explain_model).fit(X_fit, y_fit)
valid_prob = threshold_model.predict_proba(X_valid)[:, 1]

# Illustrative costs: false negatives cost 4 units; false positives cost 1.
COST_FN, COST_FP = 4.0, 1.0
threshold_grid = np.linspace(0.05, 0.95, 91)
cost_rows = []
for threshold in threshold_grid:
    pred_valid = (valid_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, pred_valid, labels=[0, 1]).ravel()
    cost_rows.append({"threshold": threshold, "cost": COST_FN*fn + COST_FP*fp,
                      "false_negatives": fn, "false_positives": fp})
threshold_table = pd.DataFrame(cost_rows)
optimal_threshold = float(threshold_table.loc[threshold_table["cost"].idxmin(), "threshold"])
test_prob_threshold = explain_model.predict_proba(X_test)[:, 1]
test_pred_threshold = (test_prob_threshold >= optimal_threshold).astype(int)
print(f"Validation-selected threshold: {optimal_threshold:.2f}")
print("Held-out test metrics at selected threshold:")
print(classification_report(y_test, test_pred_threshold, target_names=class_names, zero_division=0))
display(threshold_table.sort_values("cost").head(10))
plt.figure(figsize=(8, 4))
plt.plot(threshold_table["threshold"], threshold_table["cost"], marker=".")
plt.axvline(optimal_threshold, linestyle="--", label=f"Selected = {optimal_threshold:.2f}")
plt.xlabel("Probability threshold"); plt.ylabel("Validation cost")
plt.title("Decision threshold selection (training validation only)")
plt.legend(); plt.show()

# 42. Model robustness under small feature perturbations
This stress test adds small Gaussian perturbations to standardized numeric test features and measures prediction stability. It is a sensitivity diagnostic, not a substitute for testing on representative real-world data.

In [ ]:
robust_scaler = StandardScaler().fit(X_train)
X_test_std = robust_scaler.transform(X_test)
robustness_rows = []
base_pred = explain_model.predict(X_test)
for noise_level in [0.0, 0.01, 0.03, 0.05, 0.10]:
    rng_robust = np.random.default_rng(RANDOM_STATE)
    perturbed_std = X_test_std + rng_robust.normal(0, noise_level, X_test_std.shape)
    perturbed = pd.DataFrame(
        robust_scaler.inverse_transform(perturbed_std),
        columns=X_test.columns, index=X_test.index
    )
    perturbed_pred = explain_model.predict(perturbed)
    robustness_rows.append({
        "noise_std_in_scaled_space": noise_level,
        "prediction_agreement": float(np.mean(perturbed_pred == base_pred)),
        "accuracy": float(accuracy_score(y_test, perturbed_pred))
    })
robustness_report = pd.DataFrame(robustness_rows)
display(robustness_report)
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=robustness_report, x="noise_std_in_scaled_space",
             y="prediction_agreement", marker="o", ax=ax)
ax.set(title="Prediction stability under feature perturbations",
       xlabel="Gaussian noise standard deviation (scaled features)",
       ylabel="Agreement with unperturbed predictions", ylim=(0, 1.02))
plt.show()

# 43. Feature selection benchmark
Compare a baseline scaled logistic-regression pipeline with a leakage-safe SelectKBest pipeline. Feature selection is fitted inside each cross-validation fold. The hold-out test set is used only for the final comparison.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_validate

feature_selection_models = {
    "All features": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "SelectKBest (top 10)": Pipeline([
        ("scale", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=min(10, X_train.shape[1]))),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
}
feature_selection_rows = []
feature_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for label, estimator in feature_selection_models.items():
    cv_result = cross_validate(estimator, X_train, y_train, cv=feature_cv,
                               scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
                               n_jobs=-1)
    fitted = clone(estimator).fit(X_train, y_train)
    feature_selection_rows.append({
        "Workflow": label,
        "CV accuracy mean": cv_result["test_accuracy"].mean(),
        "CV ROC AUC mean": cv_result["test_roc_auc"].mean(),
        "Test accuracy": accuracy_score(y_test, fitted.predict(X_test)),
        "Test ROC AUC": roc_auc_score(y_test, fitted.predict_proba(X_test)[:, 1])
    })
feature_selection_comparison = pd.DataFrame(feature_selection_rows)
display(feature_selection_comparison.style.format({c: "{:.3f}" for c in feature_selection_comparison.select_dtypes("number").columns}))


# 44. Interactive 3D robustness and model-diagnostics view
When Plotly is installed, explore the robustness results interactively. This is a compact companion visualization to the static charts above.

In [ ]:
if HAS_PLOTLY:
    import plotly.express as px
    if len(robustness_report) >= 3:
        fig = px.scatter_3d(
            robustness_report,
            x="noise_std_in_scaled_space", y="prediction_agreement",
            z="accuracy", color="noise_std_in_scaled_space",
            size="accuracy", hover_data=["prediction_agreement", "accuracy"],
            title="3D model robustness diagnostics"
        )
        fig.show()
    else:
        print("Not enough robustness observations for the 3D view.")
else:
    print("Install Plotly with %pip install plotly to enable this interactive chart.")

## Upgrade notes: explainability, decisions and robustness
- PDP/ICE plots describe model response patterns and must not be interpreted as causal effects.
- The decision threshold is selected using a validation split from training data; the test set remains held out for reporting.
- Perturbation analysis is a synthetic stress test, not a formal robustness guarantee.
- Feature selection is placed inside the cross-validation pipeline to reduce leakage.
- All optional visuals use the existing Plotly availability flag.

# 45. Advanced ensemble learning: stacking and soft voting

This section adds heterogeneous ensembles. All base estimators are trained inside the stacking cross-validation procedure; the held-out test set is used only for the final report. The optional models are selected from estimators already defined above.

In [ ]:
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Keep the ensemble compact and diverse. Every estimator is cloned by sklearn.
ensemble_candidates = []
for candidate_name in ["Logistic Regression", "Random Forest", "Extra Trees"]:
    if candidate_name in models:
        ensemble_candidates.append((candidate_name.lower().replace(" ", "_"), models[candidate_name]))

if len(ensemble_candidates) >= 2:
    stacking_model = StackingClassifier(
        estimators=ensemble_candidates,
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method="predict_proba", cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1
    )
    stacking_model.fit(X_train, y_train)
    stacking_prob = stacking_model.predict_proba(X_test)[:, 1]
    stacking_pred = stacking_model.predict(X_test)
    stacking_metrics = pd.DataFrame([{
        "Model": "Stacking ensemble", "Accuracy": accuracy_score(y_test, stacking_pred),
        "F1": f1_score(y_test, stacking_pred), "ROC AUC": roc_auc_score(y_test, stacking_prob)
    }])
    display(stacking_metrics)
else:
    print("At least two base estimators are required for stacking.")

# 46. Probability calibration and reliability diagrams

Calibration checks whether predictions made with probability *p* are correct approximately *p* of the time. The calibrated model is fitted using training data only. Brier score and log loss complement discrimination metrics such as ROC AUC.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import brier_score_loss, log_loss

calibration_base = clone(selected_model)
try:
    calibrated_model = CalibratedClassifierCV(estimator=calibration_base, method="sigmoid", cv=5)
except TypeError:  # compatibility with older scikit-learn releases
    calibrated_model = CalibratedClassifierCV(base_estimator=calibration_base, method="sigmoid", cv=5)
calibrated_model.fit(X_train, y_train)
calibrated_prob = calibrated_model.predict_proba(X_test)[:, 1]
calibration_metrics = pd.DataFrame([{
    "Model": "CV-calibrated " + selected_name,
    "Brier score (lower is better)": brier_score_loss(y_test, calibrated_prob),
    "Log loss (lower is better)": log_loss(y_test, calibrated_prob),
    "ROC AUC": roc_auc_score(y_test, calibrated_prob)
}])
display(calibration_metrics)
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(y_test, selected_probability, n_bins=10, name="Uncalibrated", ax=ax)
CalibrationDisplay.from_predictions(y_test, calibrated_prob, n_bins=10, name="Calibrated", ax=ax)
ax.set_title("Probability reliability: held-out test set")
plt.tight_layout(); plt.show()

# 47. Split-conformal prediction sets

Conformal prediction adds a set of plausible labels with a marginal coverage target. A dedicated calibration split is carved out of the training data; the test set is not used to choose the quantile. Coverage is empirical and is not a guarantee for distribution-shifted data.

In [ ]:
from sklearn.model_selection import train_test_split

X_conf_fit, X_conf_cal, y_conf_fit, y_conf_cal = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)
conformal_model = clone(selected_model).fit(X_conf_fit, y_conf_fit)
cal_probs = conformal_model.predict_proba(X_conf_cal)
classes_conf = np.asarray(conformal_model.classes_)
true_class_positions = np.array([np.where(classes_conf == label)[0][0] for label in y_conf_cal])
cal_nonconformity = 1.0 - cal_probs[np.arange(len(y_conf_cal)), true_class_positions]
alpha = 0.10
n_cal = len(cal_nonconformity)
# Finite-sample conformal quantile (clipped to the available order statistic).
rank = min(n_cal, int(np.ceil((n_cal + 1) * (1 - alpha))))
conformal_quantile = float(np.sort(cal_nonconformity)[rank - 1])
test_conf_probs = conformal_model.predict_proba(X_test)
prediction_sets = [classes_conf[test_conf_probs[i] >= (1 - conformal_quantile)].tolist()
                   for i in range(len(test_conf_probs))]
covered = np.array([label in pred_set for label, pred_set in zip(y_test.to_numpy(), prediction_sets)])
set_sizes = np.array([len(s) for s in prediction_sets])
conformal_summary = pd.DataFrame([{
    "Nominal coverage": 1-alpha, "Test empirical coverage": covered.mean(),
    "Mean prediction-set size": set_sizes.mean(), "Empty-set fraction": (set_sizes == 0).mean(),
    "Calibration examples": n_cal
}])
display(conformal_summary)
print("Example prediction sets:", prediction_sets[:10])
print("Interpretation: empty sets indicate the model assigned low conformal plausibility to every class.")

# 48. Final model comparison dashboard and export

Consolidate available model results without silently treating missing metrics as zero. The CSV is an optional local artifact.

In [ ]:
dashboard_rows = []
for label, estimator in [("CV-selected model", selected_model), ("Calibrated model", calibrated_model)]:
    p_test = estimator.predict_proba(X_test)[:, 1]
    y_hat = estimator.predict(X_test)
    dashboard_rows.append({"Model": label, "Accuracy": accuracy_score(y_test, y_hat),
                           "F1": f1_score(y_test, y_hat), "ROC AUC": roc_auc_score(y_test, p_test),
                           "Brier": brier_score_loss(y_test, p_test)})
if "stacking_model" in globals():
    p_test = stacking_model.predict_proba(X_test)[:, 1]
    y_hat = stacking_model.predict(X_test)
    dashboard_rows.append({"Model": "Stacking ensemble", "Accuracy": accuracy_score(y_test, y_hat),
                           "F1": f1_score(y_test, y_hat), "ROC AUC": roc_auc_score(y_test, p_test),
                           "Brier": brier_score_loss(y_test, p_test)})
advanced_model_dashboard = pd.DataFrame(dashboard_rows).sort_values("ROC AUC", ascending=False)
display(advanced_model_dashboard.style.format({c: "{:.4f}" for c in advanced_model_dashboard.select_dtypes(include="number").columns}))
advanced_model_dashboard.to_csv("advanced_model_dashboard.csv", index=False)
print("Saved advanced_model_dashboard.csv")

## Upgrade notes: ensemble learning, calibration and conformal prediction

- Stacking uses cross-validated out-of-fold base predictions for its final estimator.
- Calibration is trained on training folds and evaluated on the held-out test set.
- Conformal prediction uses a separate calibration partition and reports empirical test coverage; coverage can change under distribution shift.
- The notebook remains educational. The breast-cancer benchmark is not a clinical diagnostic system.

# 49. Nested cross-validation: less optimistic model evaluation

Nested cross-validation separates hyperparameter selection (inner folds) from performance estimation (outer folds). It is more computationally expensive than a single train/test split, so this section uses a compact set of representative models and search spaces. The final untouched test set remains reserved for the final evaluation.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
import pandas as pd

nested_candidates = {
    "Logistic Regression": (
        Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
        {"model__C": [0.1, 1.0, 10.0]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [120, 200], "max_depth": [None, 8],
         "min_samples_leaf": [1, 2]}
    )
}
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 1)
nested_rows = []
for candidate_name, (estimator, param_grid) in nested_candidates.items():
    fold_scores, chosen_params = [], []
    for fold_no, (fit_idx, valid_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
        X_fit, X_valid = X_train.iloc[fit_idx], X_train.iloc[valid_idx]
        y_fit, y_valid = y_train.iloc[fit_idx], y_train.iloc[valid_idx]
        search = GridSearchCV(clone(estimator), param_grid, scoring="roc_auc",
                              cv=inner_cv, n_jobs=-1, refit=True)
        search.fit(X_fit, y_fit)
        valid_probability = search.best_estimator_.predict_proba(X_valid)[:, 1]
        fold_scores.append(roc_auc_score(y_valid, valid_probability))
        chosen_params.append(search.best_params_)
    nested_rows.append({"Model": candidate_name,
                        "Nested CV ROC AUC mean": float(np.mean(fold_scores)),
                        "Nested CV ROC AUC std": float(np.std(fold_scores, ddof=1)),
                        "Outer fold scores": np.round(fold_scores, 4).tolist(),
                        "Inner best parameters": chosen_params})
nested_cv_results = pd.DataFrame(nested_rows).sort_values(
    "Nested CV ROC AUC mean", ascending=False).reset_index(drop=True)
display(nested_cv_results)
print("Interpretation: compare nested-CV estimates across models; do not tune against the outer-fold scores.")

# 50. Fairness and subgroup performance audit

This section provides a reusable group-metrics audit. For a real audit, supply a legitimate grouping column from an appropriately governed dataset and assess whether its use is lawful and appropriate. The built-in breast-cancer example has no demographic group field, so the optional demonstration below uses **synthetic groups only**; its results are not evidence of real-world fairness.

In [ ]:
def subgroup_classification_audit(y_true, y_pred, group, *, positive_label=1):
    # Summarize selection rate, TPR, FPR, accuracy and support by group.
    audit = pd.DataFrame({"y_true": np.asarray(y_true),
                          "y_pred": np.asarray(y_pred),
                          "group": np.asarray(group)})
    if not (len(audit.y_true) == len(audit.y_pred) == len(audit.group)):
        raise ValueError("y_true, y_pred and group must have equal lengths.")
    rows = []
    for group_value, part in audit.groupby("group", dropna=False):
        actual_pos = part.y_true == positive_label
        predicted_pos = part.y_pred == positive_label
        tp = int((actual_pos & predicted_pos).sum())
        fn = int((actual_pos & ~predicted_pos).sum())
        fp = int((~actual_pos & predicted_pos).sum())
        tn = int((~actual_pos & ~predicted_pos).sum())
        rows.append({"Group": str(group_value), "Support": len(part),
                     "Selection rate": float(predicted_pos.mean()),
                     "TPR": tp / (tp + fn) if tp + fn else np.nan,
                     "FPR": fp / (fp + tn) if fp + tn else np.nan,
                     "Accuracy": float((part.y_true == part.y_pred).mean())})
    return pd.DataFrame(rows).sort_values("Group").reset_index(drop=True)

# Demonstration only: randomly assigned synthetic group labels, not a protected attribute.
_demo_group_rng = np.random.default_rng(RANDOM_STATE)
synthetic_demo_groups = _demo_group_rng.choice(["Synthetic group A", "Synthetic group B"],
                                               size=len(y_test), p=[0.5, 0.5])
_demo_predictions = selected_model.predict(X_test)
synthetic_fairness_demo = subgroup_classification_audit(
    y_test, _demo_predictions, synthetic_demo_groups, positive_label=1)
display(synthetic_fairness_demo)
print("Synthetic-group demonstration only. Replace with a validated, appropriate group column for a real audit.")

# 51. Multi-objective model selection: performance versus training cost

A single metric rarely captures deployment needs. This section plots ROC-AUC against fit time and marks the Pareto-efficient models: a model is dominated if another model is at least as good on both measures and strictly better on one. Fit time depends on hardware, software versions and load.

In [ ]:
if {"ROC AUC", "Fit time (s)"}.issubset(results.columns) and not results.empty:
    pareto_table = results[["Model", "ROC AUC", "Fit time (s)"]].dropna().copy()
    is_efficient = []
    for idx, row in pareto_table.iterrows():
        others = pareto_table.drop(index=idx)
        dominated = ((others["ROC AUC"] >= row["ROC AUC"]) &
                     (others["Fit time (s)"] <= row["Fit time (s)"]) &
                     ((others["ROC AUC"] > row["ROC AUC"]) |
                      (others["Fit time (s)"] < row["Fit time (s)"]))).any()
        is_efficient.append(not dominated)
    pareto_table["Pareto efficient"] = is_efficient
    display(pareto_table.sort_values(["Pareto efficient", "ROC AUC"], ascending=[False, False]))
    fig, ax = plt.subplots(figsize=(9, 5))
    for _, row in pareto_table.iterrows():
        ax.scatter(row["Fit time (s)"], row["ROC AUC"],
                   marker="*" if row["Pareto efficient"] else "o", s=110)
        ax.annotate(row["Model"], (row["Fit time (s)"], row["ROC AUC"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.set(xlabel="Fit time (seconds; lower is better)", ylabel="ROC-AUC (higher is better)",
           title="Model performance–training cost trade-off")
    plt.tight_layout(); plt.show()
else:
    print("The existing results table does not contain the required ROC AUC and fit-time columns.")

# 52. Production-style inference interface

The function below validates incoming records against the training feature schema before prediction. It is framework-independent and can be wrapped by FastAPI, Flask or another serving layer. Do not expose a model endpoint publicly without authentication, input limits, logging controls and privacy/security review.

In [ ]:
def predict_records(records, *, estimator=None, feature_names=None):
    # Predict from a list of dictionaries or a DataFrame with the trained schema.
    estimator = selected_model if estimator is None else estimator
    feature_names = list(X.columns) if feature_names is None else list(feature_names)
    incoming = records.copy() if isinstance(records, pd.DataFrame) else pd.DataFrame(records)
    missing = [name for name in feature_names if name not in incoming.columns]
    extra = [name for name in incoming.columns if name not in feature_names]
    if missing:
        raise ValueError(f"Missing required feature(s): {missing}")
    if extra:
        raise ValueError(f"Unexpected feature(s): {extra}")
    incoming = incoming.loc[:, feature_names]
    for col in feature_names:
        incoming[col] = pd.to_numeric(incoming[col], errors="raise")
    if not np.isfinite(incoming.to_numpy(dtype=float)).all():
        raise ValueError("Input contains NaN or infinite values; clean inputs before inference.")
    labels = estimator.predict(incoming)
    output = pd.DataFrame({"prediction": labels}, index=incoming.index)
    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(incoming)
        output["predicted_probability"] = probabilities.max(axis=1)
    return output

# Smoke test with held-out rows; no external service is started.
inference_smoke_test = predict_records(X_test.iloc[:3])
display(inference_smoke_test)
print("Inference schema validation and prediction smoke test completed.")

## Enterprise upgrade notes

- Nested cross-validation gives a more defensible estimate during model selection, but it does not replace the untouched final test set.
- Subgroup metrics are descriptive diagnostics, not a complete fairness determination. The synthetic demonstration is intentionally not interpreted as a real fairness result.
- Pareto efficiency depends on the selected metrics and the machine running the notebook.
- The inference helper validates schema and numeric finiteness; production deployment still requires monitoring, security, privacy, versioning and operational controls.
- Optional packages remain optional. The new sections use the notebook's existing scikit-learn, NumPy, pandas and Matplotlib stack.

# 53. Advanced time-series backtesting and forecasting

Chronological validation is essential for forecasting. This reproducible example compares a persistence baseline with a lag-feature Random Forest using expanding-window `TimeSeriesSplit`. Replace the synthetic series with timestamped operational data for real use.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

_ts_rng = np.random.default_rng(RANDOM_STATE)
_ts_n = 420
_ts_t = np.arange(_ts_n)
_ts_dates = pd.date_range("2023-01-01", periods=_ts_n, freq="D")
_ts_values = (80 + np.linspace(0, 40, _ts_n)
              + 9*np.sin(2*np.pi*_ts_t/7)
              + 12*np.sin(2*np.pi*_ts_t/365.25)
              + _ts_rng.normal(0, 4, _ts_n))
_ts_series = pd.Series(_ts_values, index=_ts_dates, name="demand")

def make_lag_features(series, lags=(1, 7, 14), windows=(7, 14)):
    frame = pd.DataFrame(index=series.index)
    for lag in lags:
        frame[f"lag_{lag}"] = series.shift(lag)
    past = series.shift(1)
    for window in windows:
        frame[f"rolling_mean_{window}"] = past.rolling(window).mean()
        frame[f"rolling_std_{window}"] = past.rolling(window).std()
    frame["day_of_week"] = series.index.dayofweek
    frame["day_of_year"] = series.index.dayofyear
    return frame.join(series.rename("target")).dropna()

_ts_supervised = make_lag_features(_ts_series)
_ts_X = _ts_supervised.drop(columns="target")
_ts_y = _ts_supervised["target"]
_ts_rows = []
for fold, (fit_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=5).split(_ts_X), 1):
    X_fit, X_val = _ts_X.iloc[fit_idx], _ts_X.iloc[val_idx]
    y_fit, y_val = _ts_y.iloc[fit_idx], _ts_y.iloc[val_idx]
    _ts_model = RandomForestRegressor(n_estimators=140, min_samples_leaf=2,
                                      random_state=RANDOM_STATE, n_jobs=-1)
    _ts_model.fit(X_fit, y_fit)
    _ts_pred = _ts_model.predict(X_val)
    _ts_rows.append({"Fold": fold, "Persistence MAE": mean_absolute_error(y_val, X_val["lag_1"]),
                     "Random Forest MAE": mean_absolute_error(y_val, _ts_pred),
                     "Random Forest RMSE": np.sqrt(mean_squared_error(y_val, _ts_pred))})
time_series_backtest = pd.DataFrame(_ts_rows)
display(time_series_backtest)

_cut = int(len(_ts_X)*0.8)
_ts_final = RandomForestRegressor(n_estimators=180, min_samples_leaf=2,
                                  random_state=RANDOM_STATE, n_jobs=-1)
_ts_final.fit(_ts_X.iloc[:_cut], _ts_y.iloc[:_cut])
_ts_holdout_pred = _ts_final.predict(_ts_X.iloc[_cut:])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(_ts_y.index[_cut:], _ts_y.iloc[_cut:], label="Observed")
ax.plot(_ts_y.index[_cut:], _ts_holdout_pred, label="Forecast")
ax.set(title="Chronological forecasting holdout", xlabel="Date", ylabel="Demand")
ax.legend(); plt.tight_layout(); plt.show()

# 54. Bootstrap confidence intervals for model metrics

Bootstrap resampling of held-out predictions gives an approximate uncertainty interval for test metrics. It describes uncertainty on this evaluation sample, not a guarantee of future performance.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
_uq_model = selected_model if "selected_model" in globals() else best_model
_uq_prob = _uq_model.predict_proba(X_test)[:, 1]
_uq_pred = _uq_model.predict(X_test)
_uq_y = np.asarray(y_test)
_uq_rng = np.random.default_rng(RANDOM_STATE)
_uq_rows = []
for _ in range(1000):
    _idx = _uq_rng.integers(0, len(_uq_y), len(_uq_y))
    if np.unique(_uq_y[_idx]).size < 2:
        continue
    _uq_rows.append([roc_auc_score(_uq_y[_idx], _uq_prob[_idx]),
                     f1_score(_uq_y[_idx], _uq_pred[_idx], zero_division=0),
                     accuracy_score(_uq_y[_idx], _uq_pred[_idx])])
_uq_df = pd.DataFrame(_uq_rows, columns=["ROC AUC", "F1", "Accuracy"])
bootstrap_metric_intervals = pd.DataFrame({
    "Metric": _uq_df.columns,
    "Observed": [roc_auc_score(_uq_y, _uq_prob), f1_score(_uq_y, _uq_pred), accuracy_score(_uq_y, _uq_pred)],
    "95% CI lower": [_uq_df[c].quantile(.025) for c in _uq_df.columns],
    "95% CI upper": [_uq_df[c].quantile(.975) for c in _uq_df.columns]})
display(bootstrap_metric_intervals.style.format({c:"{:.3f}" for c in bootstrap_metric_intervals.select_dtypes("number").columns}))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.maximum(_uq_prob, 1-_uq_prob), bins=15, edgecolor="white")
ax.set(title="Prediction confidence distribution", xlabel="Maximum class probability", ylabel="Records")
plt.tight_layout(); plt.show()

# 55. Data contracts and automated input validation

Define expected columns, types, missingness limits and optional numeric ranges before inference. The validator returns a readable report and supports fail-fast checks.

In [ ]:
def validate_data_contract(frame, *, required_columns, numeric_columns=None,
                           max_missing_fraction=0.0, numeric_ranges=None,
                           reject_extra_columns=False):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("frame must be a pandas DataFrame")
    required_columns = list(required_columns)
    numeric_columns = list(numeric_columns) if numeric_columns is not None else []
    numeric_ranges = numeric_ranges or {}
    rows = []
    missing = [c for c in required_columns if c not in frame.columns]
    extra = [c for c in frame.columns if c not in required_columns]
    rows += [("required columns present", not missing, str(missing)),
             ("unexpected columns", not (extra and reject_extra_columns), str(extra))]
    for col in required_columns:
        if col in frame:
            frac = float(frame[col].isna().mean())
            rows.append((f"{col}: missing fraction", frac <= max_missing_fraction,
                         f"observed={frac:.3f}; allowed={max_missing_fraction:.3f}"))
    for col in numeric_columns:
        if col not in frame: continue
        converted = pd.to_numeric(frame[col], errors="coerce")
        numeric_ok = converted.notna().sum() == frame[col].notna().sum()
        finite_ok = np.isfinite(converted.dropna().to_numpy(dtype=float)).all()
        rows += [(f"{col}: numeric", bool(numeric_ok), "numeric values required"),
                 (f"{col}: finite", bool(finite_ok), "infinite values not allowed")]
        if col in numeric_ranges:
            low, high = numeric_ranges[col]
            rows.append((f"{col}: range", bool(converted.dropna().between(low, high).all()),
                         f"allowed=[{low}, {high}]"))
    return pd.DataFrame(rows, columns=["Check", "Passed", "Details"])

contract_report = validate_data_contract(X_test.head(20), required_columns=X.columns,
                                         numeric_columns=X.columns, max_missing_fraction=0,
                                         reject_extra_columns=True)
display(contract_report)
assert contract_report["Passed"].all(), "Input data contract failed"
print("Demonstration input batch passed the data contract.")

# 56. Statistical feature-drift monitoring

Compare a reference dataset with a current batch using Population Stability Index (PSI) and the two-sample Kolmogorov–Smirnov test. Drift is an investigation signal; it does not by itself prove model degradation or explain its cause.

In [ ]:
from scipy.stats import ks_2samp

def feature_drift_report(reference, current, psi_warning=0.10, psi_critical=0.25):
    shared = [c for c in reference.select_dtypes(include=np.number).columns if c in current.columns]
    rows = []
    for col in shared:
        ref = pd.to_numeric(reference[col], errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
        cur = pd.to_numeric(current[col], errors="coerce").replace([np.inf,-np.inf],np.nan).dropna()
        if len(ref)<5 or len(cur)<5: continue
        edges = np.unique(np.quantile(ref, np.linspace(0,1,11)))
        if len(edges)<3:
            psi = 0.0 if np.isclose(ref.mean(),cur.mean()) else np.nan
        else:
            edges[0], edges[-1] = -np.inf, np.inf
            a = np.histogram(ref,bins=edges)[0].astype(float)
            b = np.histogram(cur,bins=edges)[0].astype(float)
            a = np.clip(a/a.sum(),1e-6,None); b = np.clip(b/b.sum(),1e-6,None)
            psi = float(np.sum((b-a)*np.log(b/a)))
        ks = ks_2samp(ref,cur)
        alert = "critical" if pd.notna(psi) and psi>=psi_critical else ("warning" if pd.notna(psi) and psi>=psi_warning else "stable")
        rows.append({"Feature":col,"PSI":psi,"KS statistic":ks.statistic,"KS p-value":ks.pvalue,"Alert":alert})
    return pd.DataFrame(rows).sort_values("PSI",ascending=False,na_position="last").reset_index(drop=True)

_drift_rng = np.random.default_rng(RANDOM_STATE+10)
_drift_reference = X_train.copy()
_drift_current = X_test.copy()
_drift_current.iloc[:, 0] = _drift_current.iloc[:, 0] + _drift_rng.normal(0, 2.5, len(_drift_current))
feature_drift_results = feature_drift_report(_drift_reference, _drift_current)
display(feature_drift_results.head(12))
if not feature_drift_results.empty:
    fig, ax = plt.subplots(figsize=(10,4))
    ax.barh(feature_drift_results.head(12)["Feature"], feature_drift_results.head(12)["PSI"])
    ax.invert_yaxis()
    ax.axvline(.10, linestyle="--", label="Warning threshold")
    ax.axvline(.25, linestyle=":", label="Critical threshold")
    ax.set_title("Feature drift screening (synthetic shifted batch)"); ax.legend()
    plt.tight_layout(); plt.show()

## Advanced upgrade notes

- Added chronological time-series backtesting, bootstrap metric intervals, reusable data contracts and statistical drift monitoring.
- Repaired a model-comparison chart that attempted to set a non-numeric label column as an index after selecting numeric columns.
- The new demonstrations use synthetic time-series and a deliberately shifted test batch. Replace them with representative production data and calibrate alert thresholds before operational use.
- Optional packages and hardware-dependent sections in earlier chapters remain optional; run the notebook from top to bottom in a compatible environment.